In [34]:
# Import required libraries for SPARQL querying
from SPARQLWrapper import SPARQLWrapper, JSON, XML, POST, GET
import pandas as pd
import json
# Import shapely for spatial operations
from shapely.wkt import loads as wkt_loads
from shapely.geometry import Point, Polygon

In [35]:
import numpy as np
import matplotlib.pyplot as plt
import time
from shapely import wkt
from shapely.geometry import Polygon

from py_wake.deficit_models.gaussian import BastankhahGaussian
from py_wake.site import UniformSite
from py_wake.utils.gradients import autograd
from py_wake.examples.data.hornsrev1 import HornsrevV80

from topfarm.cost_models.cost_model_wrappers import CostModelComponent
from topfarm.easy_drivers import EasySGDDriver, EasyScipyOptimizeDriver
from topfarm.plotting import XYPlotComp
from topfarm.constraint_components.spacing import SpacingConstraint
from topfarm import TopFarmProblem
from topfarm.constraint_components.boundary import XYBoundaryConstraint
from topfarm.recorders import TopFarmListRecorder
from topfarm.constraint_components.constraint_aggregation import ConstraintAggregation
from topfarm.constraint_components.constraint_aggregation import DistanceConstraintAggregation

from py_wake.examples.data.iea37 import IEA37_WindTurbines, IEA37Site
from py_wake.wind_turbines import WindTurbine
from py_wake.wind_turbines.power_ct_functions import PowerCtTabular

# Add these imports
from pyproj import Proj, transform

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.img_tiles as cimgt
from datetime import datetime
import re

# Updated imports - Add Folium for better maps
import folium
from folium import plugins
import branca.colormap as cm
import os
import pickle
import random
from datetime import datetime, timedelta

In [37]:
fuseki_endpoint = "http://localhost:3030/Wind/sparql"
sparql = SPARQLWrapper(fuseki_endpoint)

In [38]:
def execute_select_query(query):
    """
    Execute a SPARQL SELECT query and return results as JSON
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.setMethod(GET)
    
    try:
        results = sparql.query().convert()
        return results
    except Exception as e:
        print(f"Error executing query: {e}")
        return None

def execute_ask_query(query):
    """
    Execute a SPARQL ASK query and return boolean result
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.setMethod(GET)
    
    try:
        results = sparql.query().convert()
        return results['boolean']
    except Exception as e:
        print(f"Error executing ASK query: {e}")
        return None

def execute_construct_query(query):
    """
    Execute a SPARQL CONSTRUCT query and return RDF results
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(XML)
    sparql.setMethod(GET)
    
    try:
        results = sparql.query().convert()
        return results
    except Exception as e:
        print(f"Error executing CONSTRUCT query: {e}")
        return None

def results_to_dataframe(results):
    """
    Convert SPARQL SELECT results to pandas DataFrame
    """
    if not results or 'results' not in results:
        return pd.DataFrame()
    
    bindings = results['results']['bindings']
    if not bindings:
        return pd.DataFrame()
    
    # Extract column names
    columns = list(bindings[0].keys())
    
    # Extract data
    data = []
    for binding in bindings:
        row = {}
        for col in columns:
            if col in binding:
                row[col] = binding[col]['value']
            else:
                row[col] = None
        data.append(row)
    
    return pd.DataFrame(data)

print("SPARQL connection setup complete!")

SPARQL connection setup complete!


In [39]:
def execute_update_query(query):
    """
    Execute a SPARQL UPDATE query (INSERT/DELETE/UPDATE operations)
    
    Args:
        query (str): The SPARQL UPDATE command as a string
        
    Returns:
        bool: True if successful, False if failed
    """
    # Set up a new SPARQLWrapper instance for updates
    update_sparql = SPARQLWrapper("http://localhost:3030/Wind/update")  # Note: using /update endpoint
    update_sparql.setQuery(query)
    update_sparql.setMethod(POST)
    
    try:
        update_sparql.query()
        print("Update query executed successfully!")
        return True
    except Exception as e:
        print(f"Error executing update query: {e}")
        return False

# Example of how to use the update function
print("Update function ready!")
print("\nExample usage:")
print("1. Create your SPARQL UPDATE command as a string")
print("2. Pass it to execute_update_query(your_update_command)")
print("3. The function returns True/False for success/failure")

Update function ready!

Example usage:
1. Create your SPARQL UPDATE command as a string
2. Pass it to execute_update_query(your_update_command)
3. The function returns True/False for success/failure


In [40]:
# Get all turbine attributes and create DataFrame
all_turbines_query = """
PREFIX turbine: <windfarm/Turbine#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?turbine ?property ?value
WHERE {
    ?turbine rdf:type turbine: .
    ?turbine ?property ?value .
    FILTER(STRSTARTS(STR(?property), "windfarm/Turbine#"))
}
ORDER BY ?turbine ?property
"""

all_results = execute_select_query(all_turbines_query)
all_data_df = results_to_dataframe(all_results)
all_data_df['property_name'] = all_data_df['property'].str.replace('windfarm/Turbine#', '')

turbines_df = all_data_df.pivot_table(
    index='turbine', 
    columns='property_name', 
    values='value', 
    aggfunc='first'
).reset_index()

In [41]:
turbines_df.loc[turbines_df['hasTurbineID'] == 'Turbine1'].columns

Index(['turbine', 'hasAerodynamicAEP', 'hasAirfoilSeries', 'hasAvailability',
       'hasBladeCost', 'hasBladeMass', 'hasBladePrebend', 'hasCOE',
       'hasCapacityFactor', 'hasControl', 'hasCutInWindSpeed',
       'hasCutOutWindSpeed', 'hasDownTime', 'hasDriveTrain',
       'hasElectricalAEP', 'hasGearboxRatio', 'hasGenEfficiency',
       'hasGeometry', 'hasHubDiameter', 'hasHubHeight', 'hasHubOverhang',
       'hasICC', 'hasInclination', 'hasMaxRotorSpeed', 'hasMaxVtip',
       'hasMinRotorSpeed', 'hasNacelleMass', 'hasNacelleUptiltAngle',
       'hasNumberOfBlades', 'hasPitchAngle', 'hasPowerOutput',
       'hasRatedAerodynamicPower', 'hasRatedElectricalPower',
       'hasRatedWindSpeed', 'hasRotorConeAngle', 'hasRotorDiameter',
       'hasRotorOrientation', 'hasRotorSolidity', 'hasShaftTiltAngle',
       'hasTowerCost', 'hasTowerMass', 'hasTurbineID', 'hasTurbineModel',
       'hasTurbinePositionX', 'hasTurbinePositionY', 'hasTurbineStatus',
       'hasTurbineType', 'hasWindClass'

In [42]:
class TurbineMapGenerator:
    def __init__(self, turbines_df, turbines_coord_file, windfarm_coord_file, output_dir=None):
        self.turbines_df = turbines_df
        self.turbines_coord_file = turbines_coord_file
        self.windfarm_coord_file = windfarm_coord_file
        
        # Set output directory
        if output_dir:
            self.output_dir = output_dir
            os.makedirs(self.output_dir, exist_ok=True)
        else:
            self.output_dir = "project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine"
            os.makedirs(self.output_dir, exist_ok=True)
        
        # Load coordinate data
        self.turbine_locations = self.load_turbine_coordinates()
        self.windfarm_polygon = self.load_windfarm_coordinates()
    
    def load_turbine_coordinates(self):
        """Load turbine coordinates from CSV file."""
        try:
            turbines = pd.read_csv(self.turbines_coord_file, header=None, names=['longitude', 'latitude'])
            print(f"Loaded {len(turbines)} turbine locations")
            return turbines
        except Exception as e:
            print(f"Error loading turbine coordinates: {e}")
            return None
    
    def load_windfarm_coordinates(self):
        """Load wind farm polygon coordinates from text file."""
        try:
            with open(self.windfarm_coord_file, 'r') as f:
                content = f.read().strip()
            
            # Extract coordinates from POLYGON format
            if content.startswith('POLYGON'):
                # Remove 'POLYGON ((' and '))'
                coords_str = content.replace('POLYGON ((', '').replace('))', '')
                coord_pairs = coords_str.split(', ')
                
                coordinates = []
                for pair in coord_pairs:
                    lon, lat = pair.split(' ')
                    coordinates.append([float(lat), float(lon)])  # Folium uses [lat, lon]
                
                print(f"Loaded wind farm polygon with {len(coordinates)} points")
                return coordinates
            else:
                print("Wind farm coordinate file format not recognized")
                return None
        except Exception as e:
            print(f"Error loading wind farm coordinates: {e}")
            return None
    
    def get_turbine_info(self, turbine_id):
        """Get turbine information from the DataFrame."""
        # Find the turbine row by ID
        turbine_row = self.turbines_df[self.turbines_df['hasTurbineID'] == f'Turbine{turbine_id}']
        
        if turbine_row.empty:
            return None
        
        turbine_data = turbine_row.iloc[0]
        
        # Extract required information with safe access and format properly
        cut_out_speed = turbine_data.get('hasCutOutWindSpeed', 'N/A')
        if cut_out_speed != 'N/A':
            try:
                cut_out_speed = f"{float(cut_out_speed):.1f} m/s"
            except:
                cut_out_speed = 'N/A'
        
        pitch_angle = turbine_data.get('hasPitchAngle', 'N/A')
        if pitch_angle != 'N/A':
            try:
                pitch_angle = f"{float(pitch_angle):.1f}°"
            except:
                pitch_angle = 'N/A'
        
        yaw_angle = turbine_data.get('hasYawAngle', 'N/A')
        if yaw_angle != 'N/A':
            try:
                yaw_angle = f"{float(yaw_angle):.1f}°"
            except:
                yaw_angle = 'N/A'
        
        # Get turbine status first
        turbine_status = turbine_data.get('hasTurbineStatus', 'N/A')
        
        # Handle power output - set to 0 if turbine is parked
        power_output = turbine_data.get('hasPowerOutput', 'N/A')
        if turbine_status.lower() == 'parked':
            power_output = '0.0 MW'
        elif power_output != 'N/A':
            try:
                power_output = f"{float(power_output):.1f} MW"
            except:
                power_output = 'N/A'
        
        info = {
            'TurbineModel': turbine_data.get('hasTurbineModel', 'N/A'),
            'CutOutWindSpeed': cut_out_speed,
            'PitchAngle': pitch_angle,
            'YawAngle': yaw_angle,
            'PowerOutput': power_output,
            'TurbineStatus': turbine_status
        }
        
        return info
    
    def create_turbine_map(self, title='Wind Farm Turbines'):
        """Create an interactive map showing all turbines with their information."""
        if self.turbine_locations is None:
            print("No turbine coordinates available")
            return None
        
        # Calculate map center
        center_lat = self.turbine_locations['latitude'].mean()
        center_lon = self.turbine_locations['longitude'].mean()
        
        # Create folium map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=12,
            tiles=None
        )
        
        # Add different map layers
        folium.TileLayer(
            tiles='OpenStreetMap',
            name='OpenStreetMap',
            overlay=False,
            control=True
        ).add_to(m)
        
        folium.TileLayer(
            tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
            attr='Esri',
            name='Satellite',
            overlay=False,
            control=True
        ).add_to(m)
        
        folium.TileLayer(
            tiles='CartoDB positron',
            name='Light',
            overlay=False,
            control=True
        ).add_to(m)
        
        # Add wind farm polygon if available
        if self.windfarm_polygon:
            folium.Polygon(
                locations=self.windfarm_polygon,
                color='blue',
                weight=3,
                fillColor='lightblue',
                fillOpacity=0.3,
                popup='Wind Farm Boundary',
                tooltip='Wind Farm Area'
            ).add_to(m)
        
        # Add turbines
        for idx, turbine in self.turbine_locations.iterrows():
            turbine_id = idx + 1  # Assuming turbine IDs start from 1
            turbine_info = self.get_turbine_info(turbine_id)
            
            if turbine_info is None:
                # Default values if no info found
                turbine_info = {
                    'TurbineModel': 'N/A',
                    'CutOutWindSpeed': 'N/A',
                    'PitchAngle': 'N/A',
                    'YawAngle': 'N/A',
                    'PowerOutput': 'N/A',
                    'TurbineStatus': 'Unknown'
                }
            
            # Determine turbine color based on status
            status = turbine_info['TurbineStatus']
            status_lower = status.lower()
            if status_lower == 'operational':
                turbine_color = 'green'
                icon_color = 'green'
            elif status_lower == 'maintenance':
                turbine_color = 'orange'
                icon_color = 'orange'
            elif status_lower in ['fault', 'error']:
                turbine_color = 'red'
                icon_color = 'red'
            elif status_lower == 'parked':
                turbine_color = 'red'
                icon_color = 'red'
            else:
                turbine_color = 'gray'
                icon_color = 'gray'
            
            # Create popup content
            popup_content = f'''
            <b style="font-size: 13px;">Turbine {turbine_id}</b><br>
            <span style="font-size: 13px;">Model: {turbine_info['TurbineModel']}</span><br>
            <span style="font-size: 13px;">Cut-out Wind Speed: {turbine_info['CutOutWindSpeed']}</span><br>
            <span style="font-size: 13px;">Pitch Angle: {turbine_info['PitchAngle']}</span><br>
            <span style="font-size: 13px;">Yaw Angle: {turbine_info['YawAngle']}</span><br>
            <span style="font-size: 13px;">Power Output: {turbine_info['PowerOutput']}</span><br>
            <span style="font-size: 13px;">Status: <span style="color: {turbine_color}; font-weight: bold;">{turbine_info['TurbineStatus']}</span></span>
            '''
            
            # Add turbine marker with circular dot
            folium.CircleMarker(
                location=[turbine['latitude'], turbine['longitude']],
                radius=8,
                popup=folium.Popup(popup_content, max_width=300),
                tooltip=f'Turbine {turbine_id} - {turbine_info["TurbineStatus"]}',
                color='white',
                weight=2,
                fillColor=turbine_color,
                fillOpacity=0.8
            ).add_to(m)
        
        # Add layer control
        folium.LayerControl(position='bottomright').add_to(m)
        
        # Add title
        title_html = f'''
        <h3 align="center" style="font-size:16px; color:white; background-color:rgba(0,0,0,0.7); padding:10px; margin:10px; border-radius:5px; text-shadow: 2px 2px 4px rgba(0,0,0,0.8);"><b>{title}</b></h3>
        '''
        m.get_root().html.add_child(folium.Element(title_html))
        
        return m
    
    def create_status_summary_map(self, title='Wind Farm Status Summary'):
        """Create a map with turbines colored by status and a summary legend."""
        if self.turbine_locations is None:
            print("No turbine coordinates available")
            return None
        
        # Calculate map center
        center_lat = self.turbine_locations['latitude'].mean()
        center_lon = self.turbine_locations['longitude'].mean()
        
        # Create folium map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=12,
            tiles=None
        )
        
        # Add map layers
        folium.TileLayer(
            tiles='OpenStreetMap',
            name='OpenStreetMap',
            overlay=False,
            control=True
        ).add_to(m)
        
        folium.TileLayer(
            tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
            attr='Esri',
            name='Satellite',
            overlay=False,
            control=True
        ).add_to(m)
        
        # Add wind farm polygon
        if self.windfarm_polygon:
            folium.Polygon(
                locations=self.windfarm_polygon,
                color='blue',
                weight=3,
                fillColor='lightblue',
                fillOpacity=0.3,
                popup='Wind Farm Boundary',
                tooltip='Wind Farm Area'
            ).add_to(m)
        
        # Count turbines by status
        status_counts = {'operational': 0, 'maintenance': 0, 'fault': 0, 'parked': 0, 'unknown': 0}
        
        # Add turbines with simplified markers
        for idx, turbine in self.turbine_locations.iterrows():
            turbine_id = idx + 1
            turbine_info = self.get_turbine_info(turbine_id)
            
            if turbine_info is None:
                status = 'unknown'
            else:
                status = turbine_info['TurbineStatus'].lower()
            
            # Count status and assign color
            if status == 'operational':
                status_counts['operational'] += 1
                color = 'green'
            elif status == 'maintenance':
                status_counts['maintenance'] += 1
                color = 'orange'
            elif status in ['fault', 'error']:
                status_counts['fault'] += 1
                color = 'red'
            elif status == 'parked':
                status_counts['parked'] += 1
                color = 'red'
            else:
                status_counts['unknown'] += 1
                color = 'gray'
            
            folium.CircleMarker(
                location=[turbine['latitude'], turbine['longitude']],
                radius=8,
                color='white',
                weight=2,
                fillColor=color,
                fillOpacity=0.8,
                popup=f'Turbine {turbine_id} - {status.title()}',
                tooltip=f'Turbine {turbine_id}'
            ).add_to(m)
        
        # Add legend with status counts
        legend_html = f'''
        <div style="position: fixed; 
                    top: 10px; right: 10px; width: 220px; height: 150px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <b>Turbine Status Summary</b><br>
        <i class="fa fa-circle" style="color:green"></i> Operational: {status_counts['operational']}<br>
        <i class="fa fa-circle" style="color:orange"></i> Maintenance: {status_counts['maintenance']}<br>
        <i class="fa fa-circle" style="color:red"></i> Fault/Error: {status_counts['fault']}<br>
        <i class="fa fa-circle" style="color:red"></i> Parked: {status_counts['parked']}<br>
        <i class="fa fa-circle" style="color:gray"></i> Unknown: {status_counts['unknown']}<br>
        <b>Total: {sum(status_counts.values())}</b>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legend_html))
        
        # Add layer control
        folium.LayerControl(position='bottomright').add_to(m)
        
        # Add title
        title_html = f'''
        <h3 align="center" style="font-size:16px; color:white; background-color:rgba(0,0,0,0.7); padding:10px; margin:10px; border-radius:5px; text-shadow: 2px 2px 4px rgba(0,0,0,0.8);"><b>{title}</b></h3>
        '''
        m.get_root().html.add_child(folium.Element(title_html))
        
        return m
    
    def save_maps(self, detailed_name='turbines_default_detailed_map.html', summary_name='turbines_status_default_summary.html'):
        """Generate and save all turbine maps with custom names."""
        print("Creating turbine maps...")
        
        # Create detailed turbine map
        detailed_map = self.create_turbine_map('Wind Farm Turbines - Detailed View')
        if detailed_map:
            detailed_filepath = os.path.join(self.output_dir, detailed_name)
            detailed_map.save(detailed_filepath)
            print(f"Detailed turbine map saved to: {detailed_filepath}")
        
        # Create status summary map
        summary_map = self.create_status_summary_map('Wind Farm Status Summary')
        if summary_map:
            summary_filepath = os.path.join(self.output_dir, summary_name)
            summary_map.save(summary_filepath)
            print(f"Status summary map saved to: {summary_filepath}")
        
        return detailed_map, summary_map

In [43]:
# Initialize the turbine map generator
turbines_coord = "project_folder/Wind/Digital_Twin/Coordinates/MD turbine coordinates.txt"
windfarm_coord = "project_folder/Wind/Digital_Twin/Coordinates/MD wind farm coordinates.txt"
output_directory = "project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine"

turbine_mapper = TurbineMapGenerator(
    turbines_df=turbines_df,
    turbines_coord_file=turbines_coord,
    windfarm_coord_file=windfarm_coord,
    output_dir=output_directory
)

Loaded 121 turbine locations
Loaded wind farm polygon with 157 points


In [44]:
# Generate and save maps with custom names
# You can specify custom names here
detailed_map_name = 'turbines_updated_detailed_map.html'  # Change this to your desired name
summary_map_name = 'turbines_status_updated_summary.html'  # Change this to your desired name

detailed_map, summary_map = turbine_mapper.save_maps(
    detailed_name=detailed_map_name,
    summary_name=summary_map_name
)

# Display sample turbine information
print("\n=== SAMPLE TURBINE INFORMATION ===")
sample_info = turbine_mapper.get_turbine_info(1)
if sample_info:
    for key, value in sample_info.items():
        print(f"{key}: {value}")
else:
    print("No turbine information found for Turbine 1")

print(f"\nAll maps saved to: {output_directory}")
print("Maps created:")
print(f"1. {detailed_map_name} - Detailed view with all turbine information")
print(f"2. {summary_map_name} - Status overview with counts")

Creating turbine maps...
Detailed turbine map saved to: project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine/turbines_updated_detailed_map.html
Status summary map saved to: project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine/turbines_status_updated_summary.html

=== SAMPLE TURBINE INFORMATION ===
TurbineModel: IEA 15 MW
CutOutWindSpeed: 25.0 m/s
PitchAngle: 90.0°
YawAngle: 217.9°
PowerOutput: 0.0 MW
TurbineStatus: Parked

All maps saved to: project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine
Maps created:
1. turbines_updated_detailed_map.html - Detailed view with all turbine information
2. turbines_status_updated_summary.html - Status overview with counts
Detailed turbine map saved to: project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine/turbines_updated_detailed_map.html
Status summary map saved to: project_folder/Wind/Digital_Twin/Response_hurricane/Output_maps_turbine/turbines_status_updated_summary.html

